In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

load_dotenv(override=True)

open_ai_key = os.getenv("OPENAI_API_KEY")
if not open_ai_key:
    raise ValueError("OPENAI_API_KEY environment variable is not set.")


MODEL = "gpt-4o-mini"
openai = OpenAI()
DB = "prices.db"

In [8]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS ticket_prices (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            city TEXT UNIQUE,
            price TEXT
        )
    """)
    
    cursor.execute("""
            CREATE TABLE IF NOT EXISTS bookings (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_name TEXT NOT NULL,
    departure_city TEXT NOT NULL,
    destination_city TEXT NOT NULL,
    departure_date TEXT NOT NULL,      -- ISO 'YYYY-MM-DD'
    return_date TEXT,                  -- NULL if one-way
    ticket_type TEXT NOT NULL CHECK (ticket_type IN ('oneway', 'return')),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
        """)
    
    for city, price in ticket_prices.items():
        cursor.execute("INSERT INTO ticket_prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = excluded.price", (city, price))
    
    conn.commit()

In [13]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM ticket_prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [14]:
def create_booking(customer_name, departure_city, destination_city, departure_date, ticket_type, return_date=None):
    print(f"DATABASE TOOL CALLED: Creating booking for {customer_name}", flush=True)
    if ticket_type == "return" and not return_date:
        return "Error: Return date is required for return tickets."
    
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO bookings (customer_name, departure_city, destination_city, departure_date, return_date, ticket_type)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (customer_name, departure_city, destination_city, departure_date, return_date, ticket_type))
        conn.commit()
        return f"Booking confirmed! Reference #{cursor.lastrowid} for {customer_name}: {departure_city} -> {destination_city} ({ticket_type})."

In [15]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

booking_function = {
    "name": "create_booking",
    "description": "Create a flight booking once all required details have been collected from the customer.",
    "parameters": {
        "type": "object",
        "properties": {
            "customer_name": {"type": "string", "description": "Full name of the customer"},
            "departure_city": {"type": "string", "description": "City the customer is departing from"},
            "destination_city": {"type": "string", "description": "City the customer is traveling to"},
            "departure_date": {"type": "string", "description": "Departure date, ISO format YYYY-MM-DD"},
            "ticket_type": {"type": "string", "enum": ["oneway", "return"], "description": "Whether it's a one-way or return ticket"},
            "return_date": {"type": "string", "description": "Return date, ISO format YYYY-MM-DD. Required only if ticket_type is 'return'."}
        },
        "required": ["customer_name", "departure_city", "destination_city", "departure_date", "ticket_type"],
        "additionalProperties": False
    }
}


tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": booking_function},
]

tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'create_booking',
   'description': 'Create a flight booking once all required details have been collected from the customer.',
   'parameters': {'type': 'object',
    'properties': {'customer_name': {'type': 'string',
      'description': 'Full name of the customer'},
     'departure_city': {'type': 'string',
      'description': 'City the customer is departing from'},
     'destination_city': {'type': 'string',
      'description': 'City the customer is traveling to'},
     'departure_date': {'type': 'string',
      'description': 'Departure date, ISO format YY

In [17]:
system_message = """You are a helpful assistant for an Aieline company called FlightAI.
Give short, courteous answers to questions about flight prices and availability.
Always be accurate. If you don't know the answer, say "I don't know" instead of making up an answer.
You have access to a database of flight prices and availability.
When a customer wants to book a flight, collect: their name, departure city, destination city, travel date, and whether it's one-way or return (and the return date if return). Ask follow-up questions for anything missing before calling create_booking. Confirm details back to the customer before booking if unsure.
"""

In [18]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        args = json.loads(tool_call.function.arguments)
        name = tool_call.function.name
        if name == "get_ticket_price":
            result = get_ticket_price(args["destination_city"])
        elif name == "create_booking":
            result = create_booking(**args)
        else:
            result = "Unknown tool"
        responses.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
    return responses

In [19]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [20]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Creating booking for Adil Abbas
DATABASE TOOL CALLED: Getting price for London
DATABASE TOOL CALLED: Creating booking for Adil Abbas
